# NLP Masterclass 04: Sequence-to-Sequence, Attention, Decoding & Evaluation

**Track:** Natural Language Processing (0 to Masterclass)  
**Prerequisites:** NLP 03 (RNNs & LSTMs)  
**Difficulty:** ⭐⭐⭐ Advanced  
**Estimated Time:** 120–150 minutes

---



## 🎓 Socratic Deep Check

> *"The Bahdanau Attention mechanism (2014) allowed the decoder to 'look back' at the encoder. The Transformer (2017) removed the encoder's recurrence entirely, using ONLY attention. What made the authors believe that attention alone was sufficient — what proof did they have?"*

---

## The Evolutionary Bridge: RNN → Attention → Transformer

This notebook is the **missing link** between RNNs (NLP 03) and Transformers (DL 07).

| Year | Architecture | Limitation Solved |
|------|-------------|------------------|
| 2014 | **Seq2Seq** (Sutskever) | Variable-length input → variable-length output |
| 2015 | **+ Attention** (Bahdanau) | Bottleneck of fixed-size context vector |
| 2016 | **Beam Search** (Wu et al.) | Greedy decoding dead-ends |
| 2017 | **Transformer** (Vaswani) | Sequential processing bottleneck |



---

## 📑 Table of Contents

* [🎓 Socratic Deep Check](#-socratic-deep-check)
  * [The Evolutionary Bridge: RNN → Attention → Transformer](#the-evolutionary-bridge-rnn-attention-transformer)
* [1 · The Encoder-Decoder Framework](#1-the-encoder-decoder-framework)
* [2 · Bahdanau Attention from Scratch](#2-bahdanau-attention-from-scratch)
* [3 · Full Seq2Seq with Attention](#3-full-seq2seq-with-attention)
* [4 · Decoding Strategies](#4-decoding-strategies)
  * [4.1 · Greedy vs. Beam Search (Directed Generation)](#41-greedy-vs-beam-search-directed-generation)
  * [4.2 · Stochastic Decoding (Open-Ended Generation)](#42-stochastic-decoding-open-ended-generation)
* [5 · Text Generation Metrics: BLEU & ROUGE](#5-text-generation-metrics-bleu-rouge)
  * [5.1 · BLEU — Bilingual Evaluation Understudy](#51-bleu-bilingual-evaluation-understudy)
  * [5.2 · ROUGE — Recall-Oriented Understudy for Gisting Evaluation](#52-rouge-recall-oriented-understudy-for-gisting-evaluation)
* [6 · From RNN-Attention to Self-Attention](#6-from-rnn-attention-to-self-attention)
* [✅ Knowledge Check](#-knowledge-check)
* [🔨 Practical Practice](#-practical-practice)

---


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors see attention as "just another neural network component." Seniors understand that attention is a **differentiable database query** — Query finds relevant Keys, retrieves corresponding Values. This same abstraction appears in RAG (NB28: query=user question, keys/values=document chunks) and Agentic tool selection (NB30: query=task, keys=tool descriptions).

**Mental Model:** Imagine translating a 100-word English sentence to French. Without attention, the encoder must compress the entire sentence into ONE vector (the context). That's like reading a book, then trying to recite it from a single sticky note. With attention, the translator can look back at any word at any time — like keeping the original book open while translating.

**Common Junior Pitfall:** Thinking attention is only for NLP. Attention is used in computer vision (ViT), protein folding (AlphaFold), music generation, and graph neural networks.

---

In [ ]:
!pip install -q torch numpy matplotlib nltk rouge-score

## 1 · The Encoder-Decoder Framework

**Problem:** Map sequences of different lengths.
- Translation: "Hello world" (2 words) → "Bonjour le monde" (3 words)
- Summarization: 500 words → 50 words
- Chatbots: Question → Answer

```
ENCODER                              DECODER
[Hello] → [RNN] → h₁ ─┐             ┌─ h₁' → [RNN] → "Bonjour"
[world] → [RNN] → h₂ ─┤  context    ├─ h₂' → [RNN] → "le"
                       └──→ c ───→──┘  h₃' → [RNN] → "monde"
                      (bottleneck!)     h₄' → [RNN] → <EOS>
```

### The Bottleneck Problem
ALL information from the encoder must pass through ONE vector (`c`). For long sequences, this vector can't hold enough information.

In [ ]:
import torch
import torch.nn as nn

class Seq2SeqEncoder(nn.Module):
    """Encoder: reads input sequence, produces hidden states."""
    
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
    
    def forward(self, x):
        embedded = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(embedded)
        # outputs: ALL hidden states (seq_len, hidden_dim)
        # hidden: LAST hidden state (the "context" vector)
        return outputs, hidden, cell

# Demo
encoder = Seq2SeqEncoder(vocab_size=5000, embed_dim=64, hidden_dim=128)
src = torch.randint(0, 5000, (2, 8))  # Batch=2, source seq_len=8
enc_outputs, enc_hidden, enc_cell = encoder(src)

print(f'Source sequence:  {src.shape}')
print(f'Encoder outputs:  {enc_outputs.shape}  (ALL 8 hidden states)')
print(f'Encoder hidden:   {enc_hidden.shape}  (LAST hidden state = context)')
print(f'\nWithout attention: decoder only gets enc_hidden (128 numbers for 8 words!)')
print(f'With attention: decoder can access ALL enc_outputs at every step')

---
## 2 · Bahdanau Attention from Scratch

### The Key Insight (Bahdanau et al., 2015)

Instead of compressing the entire input into one vector, let the decoder **dynamically decide which encoder states to focus on** at each decoding step.

$$\text{score}(h_t^{dec}, h_j^{enc}) = v^T \tanh(W_1 h_t^{dec} + W_2 h_j^{enc})$$
$$\alpha_{t,j} = \text{softmax}(\text{score}_{t}) \quad \text{(attention weights)}$$
$$c_t = \sum_j \alpha_{t,j} h_j^{enc} \quad \text{(context vector — different for each decoder step!)}$$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BahdanauAttention(nn.Module):
    """Additive (Bahdanau) attention mechanism.
    
    This is the original attention that inspired the Transformer.
    The self-attention in DL 07 is the Scaled Dot-Product variant.
    """
    
    def __init__(self, hidden_dim):
        super().__init__()
        self.W1 = nn.Linear(hidden_dim, hidden_dim)  # For decoder state
        self.W2 = nn.Linear(hidden_dim, hidden_dim)  # For encoder outputs
        self.v = nn.Linear(hidden_dim, 1)             # Score vector
    
    def forward(self, decoder_hidden, encoder_outputs):
        """
        Args:
            decoder_hidden: (batch, hidden_dim) — current decoder state
            encoder_outputs: (batch, src_len, hidden_dim) — all encoder states
        Returns:
            context: (batch, hidden_dim) — weighted sum of encoder states
            weights: (batch, src_len) — attention distribution
        """
        # decoder_hidden: (batch, hidden) -> (batch, 1, hidden) for broadcasting
        dec_expanded = decoder_hidden.unsqueeze(1)
        
        # Score: how relevant is each encoder state to the current decoder state?
        scores = self.v(torch.tanh(
            self.W1(dec_expanded) + self.W2(encoder_outputs)
        )).squeeze(-1)  # (batch, src_len)
        
        # Attention weights (softmax over source positions)
        weights = F.softmax(scores, dim=-1)  # (batch, src_len)
        
        # Context: weighted combination of encoder states
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)
        
        return context, weights

# Demo: Attention at one decoder step
attn = BahdanauAttention(hidden_dim=128)
dec_h = torch.randn(2, 128)      # Current decoder state
enc_outs = torch.randn(2, 8, 128)  # 8 encoder states

context, weights = attn(dec_h, enc_outs)

print('Bahdanau Attention Demo:')
print(f'  Decoder state:     {dec_h.shape}')
print(f'  Encoder states:    {enc_outs.shape}')
print(f'  Context vector:    {context.shape}  (weighted mix of encoder states)')
print(f'  Attention weights: {weights.shape}')
print(f'\n  Weights for batch 0: {weights[0].detach().numpy().round(3)}')
print(f'  Sum = {weights[0].sum().item():.1f} (probability distribution)')
print(f'\n  Highest attention on position {weights[0].argmax().item()}')
print(f'  → Decoder is "looking at" encoder step {weights[0].argmax().item()}')

---
## 3 · Full Seq2Seq with Attention

In [ ]:
class AttentionDecoder(nn.Module):
    """Decoder that uses attention to access all encoder states."""
    
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attention = BahdanauAttention(hidden_dim)
        self.lstm = nn.LSTM(embed_dim + hidden_dim, hidden_dim, batch_first=True)
        self.output_proj = nn.Linear(hidden_dim, vocab_size)
    
    def forward_step(self, token, hidden, cell, encoder_outputs):
        """One step of decoding."""
        embedded = self.embedding(token)  # (batch, embed_dim)
        
        # Attention: which encoder states matter now?
        context, attn_weights = self.attention(hidden.squeeze(0), encoder_outputs)
        
        # Concatenate context + embedded input
        lstm_input = torch.cat([embedded, context], dim=1).unsqueeze(1)
        
        # LSTM step
        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))
        
        # Predict next token
        prediction = self.output_proj(output.squeeze(1))
        
        return prediction, hidden, cell, attn_weights

# Full Seq2Seq with Attention
encoder = Seq2SeqEncoder(5000, 64, 128)
decoder = AttentionDecoder(5000, 64, 128)

# Encode source
src = torch.randint(0, 5000, (2, 8))
enc_outputs, enc_hidden, enc_cell = encoder(src)

# Decode step by step
current_token = torch.zeros(2, dtype=torch.long)  # <SOS> token
hidden, cell = enc_hidden, enc_cell

print('Attention during translation (batch 0):')
src_words = ['The', 'cat', 'sat', 'on', 'the', 'mat', '.', '<pad>']

for step in range(4):
    pred, hidden, cell, attn_w = decoder.forward_step(
        current_token, hidden, cell, enc_outputs
    )
    current_token = pred.argmax(dim=1)
    weights = attn_w[0].detach().numpy()
    top_idx = weights.argmax()
    print(f'  Step {step}: attending to "{src_words[top_idx]}" (weight={weights[top_idx]:.3f})')

---
* [4 · Decoding Strategies](#4-decoding-strategies)
  * [4.1 · Greedy vs. Beam Search (Directed Generation)](#41-greedy-vs-beam-search-directed-generation)
  * [4.2 · Stochastic Decoding (Open-Ended Generation)](#42-stochastic-decoding-open-ended-generation)

### The Problem with Greedy Decoding

At each step, greedy decoding picks the single most probable token:
$$\hat{y}_t = \arg\max_w P(w \mid \hat{y}_{<t}, x)$$

**Why is this wrong?** Consider the probability landscape:

```
Step 1    Step 2    Step 3    Joint Probability
"The"  ──▶ "cat"  ──▶ "sat"   P = 0.5 × 0.6 × 0.7 = 0.210  ✓ Greedy picks this
"A"    ──▶ "large"──▶ "black" P = 0.4 × 0.9 × 0.9 = 0.324  ✗ Greedy never explores
```

Greedy decoding commits to "The" (p=0.5) at step 1, never discovering that the globally
better sequence starts with "A" (p=0.4). **A locally optimal choice leads to a globally
suboptimal sequence** — the classic greedy algorithm pitfall.

### Beam Search: Maintaining k Hypotheses

Beam Search keeps the **k best partial sequences** (the "beam") alive at each step.
Instead of one path, it explores a frontier of width `k`.

**Algorithm (beam width = k):**
1. Start with `k=1` hypothesis: `[<SOS>]` with log-prob = 0
2. At each step, expand every hypothesis by **all** vocabulary tokens
3. Score each candidate: `log P(sequence) = sum of log P(each token)`
4. Keep only the **top k** candidates → the new beam
5. Stop when all k beams hit `<EOS>` or max length is reached
6. Return the highest-scoring completed sequence

**Why log probabilities?** Multiplying many small probabilities causes numerical underflow
(`0.5^50 ≈ 10⁻¹⁵`). Log-space converts multiplication to addition: `log(p₁·p₂) = log p₁ + log p₂`.

**Length Penalty:** Beam Search naturally favours shorter sequences (fewer multiplications
of values < 1). Apply a length penalty to normalize:
$$\text{score} = \frac{\log P(y)}{|y|^\alpha}, \quad \alpha \approx 0.6\text{–}0.7$$

In [ ]:
import math
from typing import List, Tuple, Optional

# ── Toy vocabulary for demonstration ──────────────────────────────────────────
VOCAB = {
    '<SOS>': 0, '<EOS>': 1, '<PAD>': 2,
    'The': 3, 'A': 4, 'cat': 5, 'dog': 6,
    'large': 7, 'small': 8, 'sat': 9, 'ran': 10,
    'black': 11, 'white': 12
}
ID2WORD = {v: k for k, v in VOCAB.items()}
VOCAB_SIZE = len(VOCAB)
EOS_ID = VOCAB['<EOS>']

# ── Toy language model: returns log-probs over vocabulary ─────────────────────
def toy_lm(sequence: List[int]) -> List[float]:
    """A hand-crafted toy LM to illustrate beam search.
    
    Returns log-probabilities for the next token given the sequence so far.
    In a real system this would be a neural network forward pass.
    """
    # Define conditional log-prob tables (manually crafted for the demo)
    tables = {
        # P(word | <SOS>)
        (0,): { 3: math.log(0.50),  4: math.log(0.40),  1: math.log(0.10) },   # The=0.5 A=0.4
        # P(word | The)
        (0,3): { 5: math.log(0.60), 6: math.log(0.30),  1: math.log(0.10) },   # cat=0.6 dog=0.3
        # P(word | A)
        (0,4): { 7: math.log(0.90), 8: math.log(0.05),  1: math.log(0.05) },   # large=0.9!
        # P(word | The cat)
        (0,3,5): { 9: math.log(0.70), 10: math.log(0.20),  1: math.log(0.10) },
        # P(word | The dog)
        (0,3,6): { 10: math.log(0.80), 9: math.log(0.10),  1: math.log(0.10) },
        # P(word | A large)
        (0,4,7): { 11: math.log(0.90), 12: math.log(0.05), 1: math.log(0.05) },
        # P(word | A small)
        (0,4,8): { 12: math.log(0.80), 11: math.log(0.10), 1: math.log(0.10) },
    }
    key = tuple(sequence)
    if key in tables:
        row = tables[key]
    else:
        # Default: uniform + slight EOS boost
        row = {i: math.log(1.0 / VOCAB_SIZE) for i in range(VOCAB_SIZE)}
        row[EOS_ID] = math.log(0.9)
    # Fill any missing tokens with a tiny probability
    return [row.get(i, math.log(1e-9)) for i in range(VOCAB_SIZE)]


# ── Greedy Decoding ────────────────────────────────────────────────────────────
def greedy_decode(max_len: int = 5) -> Tuple[List[int], float]:
    """Generate a sequence by always picking the most probable next token."""
    sequence = [VOCAB['<SOS>']]
    total_log_prob = 0.0
    for _ in range(max_len):
        log_probs = toy_lm(sequence)
        best_token = max(range(VOCAB_SIZE), key=lambda t: log_probs[t])
        total_log_prob += log_probs[best_token]
        sequence.append(best_token)
        if best_token == EOS_ID:
            break
    return sequence[1:], total_log_prob   # strip <SOS>


# ── Beam Search ───────────────────────────────────────────────────────────────
def beam_search(
    beam_width: int = 3,
    max_len: int = 5,
    length_penalty_alpha: float = 0.6
) -> List[Tuple[List[int], float]]:
    """Beam Search decoding.

    Each hypothesis is (token_ids, cumulative_log_prob).
    Returns top-k completed hypotheses sorted by length-penalised score.

    Args:
        beam_width: Number of hypotheses to keep at each step (k).
        max_len: Maximum output tokens.
        length_penalty_alpha: Exponent for length normalisation (0 = no penalty).
    """
    # Each beam entry: (token_sequence_including_SOS, cumulative_log_prob)
    beams: List[Tuple[List[int], float]] = [([VOCAB['<SOS>']], 0.0)]
    completed: List[Tuple[List[int], float]] = []

    for step in range(max_len):
        all_candidates: List[Tuple[List[int], float]] = []

        for seq, cum_lp in beams:
            if seq[-1] == EOS_ID:
                completed.append((seq, cum_lp))
                continue  # Do not expand finished hypotheses

            log_probs = toy_lm(seq)

            # Expand: consider every token in the vocabulary
            for token_id, lp in enumerate(log_probs):
                candidate_lp = cum_lp + lp
                all_candidates.append((seq + [token_id], candidate_lp))

        if not all_candidates:
            break

        # Keep top-k by raw log-prob (length penalty applied at the end)
        all_candidates.sort(key=lambda x: x[1], reverse=True)
        beams = all_candidates[:beam_width]

    # Collect any still-open beams as completed
    completed.extend(beams)

    # Apply length penalty and sort
    def penalised_score(seq_lp: Tuple[List[int], float]) -> float:
        seq, lp = seq_lp
        length = max(len(seq) - 1, 1)  # exclude <SOS>
        return lp / (length ** length_penalty_alpha)

    completed.sort(key=penalised_score, reverse=True)
    return completed


# ── Compare Greedy vs. Beam Search ────────────────────────────────────────────
print('=' * 60)
print('GREEDY DECODING')
print('=' * 60)
greedy_seq, greedy_lp = greedy_decode()
greedy_words = [ID2WORD.get(t, '?') for t in greedy_seq]
print(f'  Sequence : {greedy_words}')
print(f'  Log-Prob : {greedy_lp:.4f}  (joint log-probability)')
print(f'  Exp-Prob : {math.exp(greedy_lp):.6f}')

print()
print('=' * 60)
print('BEAM SEARCH (k=3)')
print('=' * 60)
beam_results = beam_search(beam_width=3)
for rank, (seq, lp) in enumerate(beam_results[:5], 1):
    words = [ID2WORD.get(t, '?') for t in seq[1:]]  # strip <SOS>
    print(f'  #{rank}  score={lp:.4f}  ({math.exp(lp):.6f})  → {words}')

print()
best_beam_words = [ID2WORD.get(t, '?') for t in beam_results[0][0][1:]]
print(f'Greedy best : {greedy_words}  (log-prob = {greedy_lp:.4f})')
print(f'Beam   best : {best_beam_words}  (log-prob = {beam_results[0][1]:.4f})')
print()
print('➤ Beam Search found the globally better sequence that Greedy missed!')

### 🎓 Key Takeaways: Beam Search in Production

| Property | Greedy (k=1) | Beam Search (k>1) |
|----------|-------------|-------------------|
| Speed | O(n·V) | O(n·k·V) |
| Memory | O(n) | O(k·n) |
| Quality | Suboptimal | Closer to global optimum |
| Diversity | Zero — one path | Low — k similar paths |
| Typical k | — | 4 (MT), 5 (summarisation) |

**Production note:** Beam Search is the workhorse of MT systems (Google Translate, DeepL).
Modern LLMs often use **top-p / top-k sampling** with temperature instead, prioritising
diversity over determinism. The `generate()` method in HuggingFace `transformers` exposes
all of these strategies under one API.

---

### 4.2 · Stochastic Decoding (Open-Ended Generation)

**Directed vs. Open-Ended Generation**
- **Directed Generation** (Translation, Summarization): The goal is to find the single *best* output that faithfully reflects the input. In this domain, **Beam Search rules** because the probability distribution is highly constrained.
- **Open-Ended Generation** (Storytelling, Chatbots, Code Generation): The goal is creativity, fluidity, and continuation. If you apply Beam Search here, the model often falls into **boring, repetitive loops** (e.g., *"I don't know, I don't know, I don't know..."*). Why? Because highly repetitive text is structurally "safe" and receives high probability from the model. To generate human-like text, we must introduce controlled randomness.

Instead of picking the argmax, we sample from the probability distribution $P(w \mid y_{<t})$. But sampling purely from the raw softmax distribution is dangerous: the "long tail" of low-probability, nonsensical tokens accumulates enough mass that the model will eventually sample gibberish.

#### Top-k Sampling
To solve the long-tail problem, **Top-k Sampling** truncates the vocabulary to only the $k$ most likely tokens. The probabilities of these $k$ tokens are then renormalized to sum to 1, and we sample from this restricted set.

Let $V^{(k)}$ be the subset of the vocabulary containing the $k$ tokens with the highest probabilities. We define the new distribution $P'$ as:

$$P'(w_i) = \begin{cases} \frac{P(w_i)}{\sum_{w_x \in V^{(k)}} P(w_x)} & \text{if } w_i \in V^{(k)} \\ 0 & \text{otherwise} \end{cases}$$

**The Flaw of Top-k:** Fixing $k$ is problematic because the shape of the probability distribution changes dramatically between steps. 
- If the distribution is **peaked** (e.g., grammar forces a specific part of speech), picking from $k=50$ might include irrelevant words.
- If the distribution is **flat** (e.g., starting a new paragraph), $k=50$ might artificially restrict creativity.

#### Top-p (Nucleus) Sampling
**Nucleus Sampling** (Holtzman et al., 2019) elegantly solves the dynamic distribution problem by selecting the smallest set of tokens $V^{(p)}$ whose cumulative probability exceeds a threshold $p$ (e.g., $p=0.9$).

$$V^{(p)} = \arg\min_{V'} \left( \sum_{w \in V'} P(w) \geq p \right)$$

This means the effective $k$ expands and contracts organically:
- **Peaked distribution:** Top 2 tokens sum to $0.95$. Effective $k=2$.
- **Flat distribution:** Top 100 tokens sum to $0.90$. Effective $k=100$.

Let's implement the core operations required for Nucleus Sampling from scratch.


In [ ]:
import torch
import torch.nn.functional as F

def nucleus_sample(logits: torch.Tensor, p: float = 0.9, temperature: float = 1.0) -> torch.Tensor:
    """Performs Top-p (Nucleus) sampling from raw logits.
    
    Args:
        logits: (batch_size, vocab_size) - raw unnormalized scores from the model.
        p: Cumulative probability threshold (0.0 to 1.0).
        temperature: Softmax temperature (T > 1 flattens, T < 1 peaks).
        
    Returns:
        sampled_token_ids: (batch_size, 1) - the selected tokens.
    """
    # 1. Apply temperature scaling
    logits = logits / temperature
    
    # 2. Sort the logits in descending order
    sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
    
    # 3. Compute softmax probabilities and cumulative probabilities on the sorted tensor
    sorted_probs = F.softmax(sorted_logits, dim=-1)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
    
    # 4. Create a boolean mask to filter out tokens beyond the threshold
    # We shift the mask by one to ensure the token that crosses the threshold is INCLUDED.
    cutoff_mask = cumulative_probs > p
    cutoff_mask[..., 1:] = cutoff_mask[..., :-1].clone()  # Shift right
    cutoff_mask[..., 0] = 0                               # Always keep the highest prob token
    
    # 5. Apply the mask by setting rejected token logits to -infinity
    # Scatter the mask back to the original index order to apply it to unsorted logits
    indices_to_remove = cutoff_mask.scatter(dim=-1, index=sorted_indices, src=cutoff_mask)
    filtered_logits = logits.masked_fill(indices_to_remove, float('-inf'))
    
    # 6. Sample from the filtered distribution
    filtered_probs = F.softmax(filtered_logits, dim=-1)
    sampled_tokens = torch.multinomial(filtered_probs, num_samples=1)
    
    return sampled_tokens

# ── Demonstration ─────────────────────────────────────────────────────────────
torch.manual_seed(42)

# Mock logits for a vocabulary of 10 tokens (batch_size=1)
mock_logits = torch.tensor([[5.0, 4.0, 3.5, 2.0, 1.0, 0.5, 0.2, 0.1, -1.0, -2.0]])
raw_probs = F.softmax(mock_logits, dim=-1)

print("Vocabulary Probabilities (sorted):")
sorted_p, _ = torch.sort(raw_probs, descending=True)
for i, prob in enumerate(sorted_p[0].tolist()):
    print(f"  Rank {i+1}: {prob:.4f}")

print("\nSampling 10 times with p=0.9 (Nucleus Sampling):")
# With p=0.9, we keep tokens whose cumulative prob <= 0.90 + the threshold token
samples = [nucleus_sample(mock_logits, p=0.90).item() for _ in range(10)]
print(f"  Resulting token IDs: {samples}")
print("  Notice how lower-ranked tokens are safely excluded.")


---


## 5 · Text Generation Metrics: BLEU & ROUGE

Once we can generate sequences, we need to **measure quality**. Human evaluation is gold
standard but expensive. Automatic metrics are fast, reproducible, and correlate well with
human judgement at the corpus level.

### 5.1 · BLEU — Bilingual Evaluation Understudy

**Designed for:** Machine translation (one-to-one mapping of meaning)  
**Core idea:** Precision — how many n-grams in the *hypothesis* appear in the *reference*?

$$\text{BLEU} = BP \cdot \exp\left(\sum_{n=1}^{N} w_n \log p_n\right)$$

Where:
- $p_n$ = **modified n-gram precision** for n-grams of size $n$:
$$p_n = \frac{\sum_{\text{hyp}} \text{Count}_{\text{clip}}(n\text{-gram})}{\sum_{\text{hyp}} \text{Count}(n\text{-gram})}$$
- $w_n = 1/N$ (uniform weights, typically $N=4$)
- **Brevity Penalty (BP):** Punishes outputs shorter than the reference.
$$BP = \begin{cases} 1 & \text{if } c > r \\ e^{1 - r/c} & \text{if } c \leq r \end{cases}$$

**Why "modified" precision?** Clip counts at the maximum reference count to prevent reward
hacking (e.g., repeating "the the the the" to inflate unigram precision).

**BLEU's known limitations:**
- **No recall** — a hypothesis that covers only one part of the reference gets no penalty.
- **No synonyms** — "automobile" and "car" score zero overlap.
- **Corpus-level** — individual sentence BLEU is noisy.

---

### 5.2 · ROUGE — Recall-Oriented Understudy for Gisting Evaluation

**Designed for:** Summarisation (a good summary *covers* the reference content)  
**Core idea:** Recall — how many n-grams in the *reference* appear in the *hypothesis*?

**ROUGE-N (n-gram recall):**
$$\text{ROUGE-N} = \frac{\sum_{\text{ref}} \text{Count}_{\text{match}}(n\text{-gram})}{\sum_{\text{ref}} \text{Count}(n\text{-gram})}$$

**ROUGE-L (Longest Common Subsequence):**  
Uses LCS instead of fixed n-grams — captures sentence-level structure even when words
appear in different order:
$$\text{ROUGE-L} = \frac{(1 + \beta^2) \cdot R_{lcs} \cdot P_{lcs}}{R_{lcs} + \beta^2 \cdot P_{lcs}}$$

Where $R_{lcs}$ = LCS recall, $P_{lcs}$ = LCS precision, $\beta$ controls the balance (usually $\beta \gg 1$ to favour recall).

### 🎓 BLEU vs. ROUGE: The Mental Model

| | BLEU | ROUGE |
|--|------|-------|
| **Orientation** | Precision | Recall |
| **Question** | "Is the output faithful?" | "Does it cover the reference?" |
| **Task** | Translation | Summarisation |
| **Danger** | Verbose output rewarded | Short output rewarded |
| **Variants** | BLEU-1 → BLEU-4 | ROUGE-1, ROUGE-2, ROUGE-L |

In [ ]:
# ── Install if not already present ───────────────────────────────────────────
# pip install nltk rouge-score

import math
from collections import Counter
from typing import List


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PART A: BLEU from scratch (corpus-level, N=4)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _ngrams(tokens: List[str], n: int) -> Counter:
    """Count n-grams in a token list."""
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))


def bleu_score_scratch(
    hypothesis: str,
    reference: str,
    max_n: int = 4
) -> float:
    """Sentence-level BLEU score (Papineni et al., 2002) from scratch.

    Args:
        hypothesis: The model-generated string.
        reference:  The ground-truth string.
        max_n:      Maximum n-gram order (default 4 = BLEU-4).

    Returns:
        BLEU score in [0, 1].
    """
    hyp_tokens = hypothesis.lower().split()
    ref_tokens = reference.lower().split()

    if len(hyp_tokens) == 0:
        return 0.0

    # Brevity Penalty
    c, r = len(hyp_tokens), len(ref_tokens)
    bp = 1.0 if c > r else math.exp(1 - r / c)

    log_avg = 0.0
    for n in range(1, max_n + 1):
        hyp_ngrams = _ngrams(hyp_tokens, n)
        ref_ngrams = _ngrams(ref_tokens, n)
        # Modified clipped precision
        clipped = sum((min(cnt, ref_ngrams[ng]) for ng, cnt in hyp_ngrams.items()))
        total   = sum(hyp_ngrams.values())
        if total == 0 or clipped == 0:
            return 0.0  # Any zero precision → BLEU = 0
        log_avg += (1.0 / max_n) * math.log(clipped / total)

    return bp * math.exp(log_avg)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PART B: ROUGE-L from scratch (LCS-based)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _lcs_length(x: List[str], y: List[str]) -> int:
    """Compute the length of the Longest Common Subsequence (dynamic programming)."""
    m, n = len(x), len(y)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if x[i-1] == y[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    return dp[m][n]


def rouge_l_scratch(
    hypothesis: str,
    reference: str,
    beta: float = 1.2
) -> dict:
    """ROUGE-L F1 score from scratch.

    Args:
        hypothesis: The model-generated string.
        reference:  The ground-truth string.
        beta:       Weight favouring recall (> 1 = recall-biased).

    Returns:
        Dict with 'precision', 'recall', 'f1' keys.
    """
    hyp_tokens = hypothesis.lower().split()
    ref_tokens = reference.lower().split()

    lcs = _lcs_length(hyp_tokens, ref_tokens)
    if lcs == 0:
        return {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}

    precision = lcs / len(hyp_tokens)
    recall    = lcs / len(ref_tokens)
    f1 = ((1 + beta**2) * precision * recall) / (precision + beta**2 * recall)
    return {'precision': precision, 'recall': recall, 'f1': f1}


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PART C: Library verification with NLTK + rouge-score
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

# ── Example sentences ────────────────────────────────────────────────────────
reference  = "The cat sat on the mat near the window"
hypothesis_close  = "The cat sat on the mat near the window"  # perfect
hypothesis_poor   = "A dog ran outside in the rain all day"   # poor
hypothesis_partial = "The cat sat on the mat"                  # partial coverage

examples = [
    ("Perfect match",   hypothesis_close),
    ("Partial coverage", hypothesis_partial),
    ("Poor match",       hypothesis_poor),
]

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
smoother = SmoothingFunction().method1  # Avoids 0 for short sentences

print(f'Reference: "{reference}"')
print(f'{"─"*70}')
print(f'{"Label":<20} {"BLEU-4 (scratch)":>18} {"BLEU-4 (NLTK)":>15} {"ROUGE-L F1":>12}')
print(f'{"─"*70}')

for label, hyp in examples:
    # Scratch implementations
    bleu_s  = bleu_score_scratch(hyp, reference)
    rouge_s = rouge_l_scratch(hyp, reference)

    # Library implementations (cross-check)
    ref_tok = [reference.lower().split()]
    hyp_tok = hyp.lower().split()
    bleu_lib   = sentence_bleu(ref_tok, hyp_tok, smoothing_function=smoother)
    rouge_lib  = scorer.score(reference, hyp)['rougeL'].fmeasure

    print(f'{label:<20} {bleu_s:>18.4f} {bleu_lib:>15.4f} {rouge_s["f1"]:>12.4f}')

print(f'{"─"*70}')
print()
print('Observations:')
print('  • BLEU (precision) drops sharply even for partial coverage.')
print('  • ROUGE-L (recall-biased) is more lenient for partial coverage.')
print('  • Scratch vs. library values agree closely (small diffs = smoothing).')

### 🎓 Junior to Senior: Metrics in the Real World

**The pitfall:** Optimising BLEU / ROUGE too aggressively causes **metric hacking** —
the model learns to produce safe, high-overlap sequences rather than creative, correct ones.
Google's production MT systems use BLEU for coarse filtering but rely on human evaluations
for major releases.

**Modern alternatives:**
- **BERTScore** (2020) — replaces exact n-gram overlap with contextual embedding cosine similarity; handles synonyms naturally.
- **MoverScore** — uses Earth Mover's Distance over token embeddings.
- **METEOR** (2005) — adds stemming and synonym matching to BLEU.
- **ChrF** — character n-gram F-score; works well for morphologically rich languages.

**Rule of thumb for choosing:**

| Task | Recommended Metric |
|------|-------------------|
| Machine Translation | BLEU-4, ChrF, COMET |
| Summarisation | ROUGE-1, ROUGE-2, ROUGE-L |
| Open-ended generation | BERTScore, human eval |
| Code generation | pass@k (execution-based) |

---

---
## 6 · From RNN-Attention to Self-Attention

### The Final Insight (Vaswani et al., 2017: "Attention Is All You Need")

| RNN + Attention (Bahdanau) | Transformer (Vaswani) |
|---------------------------|----------------------|
| Encoder uses RNN (sequential) | Encoder uses Self-Attention (parallel) |
| Attention is decoder → encoder only | Self-attention within encoder AND decoder |
| Q = decoder state, K,V = encoder states | Q, K, V all from same sequence |
| O(n) sequential steps | O(1) parallel + O(n²) attention |

### 🎓 DEEP QUESTION ANSWERED

**Q:** *What proof did the Transformer authors have that attention alone was sufficient?*

**A:** Empirical evidence showed that in RNN+Attention models, the RNN's contribution was minimal — the attention mechanism was doing most of the heavy lifting. The theory: if attention can learn arbitrary weighted combinations of input representations, and we add positional encoding to preserve order, then the RNN's recurrence is redundant. The proof was the remarkable results on WMT translation benchmarks where the recurrence-free Transformer outperformed the best RNN models while training 10x faster.

**→ For the full Transformer implementation and Self-Attention deep-dive, see DL 07.**

---

## ✅ Knowledge Check

### Q1: Bottleneck 
In vanilla Seq2Seq (no attention), what happens to translation quality as the source sentence gets longer?

<details><summary>👉 Answer</summary>

Quality degrades dramatically. The fixed-size context vector cannot compress more than ~20-30 words of information effectively. Empirically, BLEU scores drop sharply for sentences longer than 20 words. This is the exact problem that attention solves.
</details>

### Q2: Beam Search trade-off
Why does increasing beam width `k` not always improve translation quality beyond a certain point?

<details><summary>👉 Answer</summary>

Beyond k≈5, the marginal sequences in the beam are very similar to the top-1 output (the model's probability distribution is sharply peaked). The search space explored is already close to optimal. Additionally, very large beams with length penalty can favour unnaturally short outputs. Empirically, BLEU peaks around k=4–8 and then plateaus or declines.
</details>

### Q3: BLEU vs. ROUGE
You build a summarisation system. A colleague says "just use BLEU to evaluate it." Why is this a mistake?

<details><summary>👉 Answer</summary>

BLEU measures precision — whether the words in the hypothesis appear in the reference. For summarisation, the critical property is recall — does the summary cover the key content from the reference? A high-BLEU summary could be very short and precise but miss critical information. ROUGE-L or ROUGE-2 are the standard choices for summarisation evaluation.
</details>

### Q4: Attention as search
How is Bahdanau attention similar to what happens in RAG retrieval (NB28)?

<details><summary>👉 Answer</summary>

Both are query-based retrieval: In attention, the decoder state is the query and encoder states are the documents. In RAG, the user question embedding is the query and document chunk embeddings are the Keys. Both compute similarity scores, apply softmax, and return a weighted combination. RAG is literally attention applied to a document database.
</details>

---

## 🔨 Practical Practice

### Exercise 1: Attention Visualization
Modify the demo to accumulate attention weights across all decoder steps. Plot a heatmap with source tokens on the x-axis and target steps on the y-axis.

### Exercise 2: Luong Attention
Implement Luong (dot-product) attention: $\text{score} = h_t^{dec} \cdot h_j^{enc}$. Compare parameter counts with Bahdanau.

### Exercise 3: Teacher Forcing
Implement teacher forcing in the decoder: during training, feed the ground-truth previous token instead of the model's prediction. Why does this speed up training but can hurt inference?

### Exercise 4: Diverse Beam Search
Standard beam search produces k very similar outputs. Research "Diverse Beam Search" (Vijayakumar et al., 2016). What penalty term does it add, and why does diversity matter for summarisation?

### Exercise 5: BERTScore
Install `bert-score` and compute BERTScore for the three hypothesis examples above. Which metric (BLEU, ROUGE-L, BERTScore) best captures the semantic quality of each hypothesis?

---

**Next →** NLP 05: Pre-training Objectives & BERT